# MVP Java Migration System

A clean, granular Java 8→21 migration system with prompts/ integration and step-by-step control.

In [ ]:
# Install required dependencies
!uv pip install pyyaml requests packaging

In [ ]:
import os
import sys
import json
import yaml
import subprocess
from typing import Dict, List, Any, Optional
from pathlib import Path
from dataclasses import dataclass
from datetime import datetime
import xml.etree.ElementTree as ET

# Import our prompt manager
sys.path.append('.')
from prompts.prompt_manager import PromptManager

print("✅ Imports loaded successfully!")

In [ ]:
@dataclass
class MigrationState:
    """Track migration progress and results"""
    project_path: str
    current_step: str = "initialization"
    issues_found: List[str] = None
    fixes_applied: List[str] = None
    errors: List[str] = None
    success: bool = False
    
    def __post_init__(self):
        if self.issues_found is None:
            self.issues_found = []
        if self.fixes_applied is None:
            self.fixes_applied = []
        if self.errors is None:
            self.errors = []

# Initialize project configuration
PROJECT_PATH = "/Users/abhisheksankar/Desktop/PyTorch-Notebooks/test/xsync"
migration_state = MigrationState(project_path=PROJECT_PATH)

# Initialize prompt manager
prompt_manager = PromptManager('prompts')

print(f"🎯 Migration system initialized for: {migration_state.project_path}")
print(f"📋 Available prompt templates: {', '.join(prompt_manager.list_templates())}")

## Step 1: Project Analysis

Analyze the Java project for migration opportunities and current state.

In [ ]:
def analyze_project() -> Dict[str, Any]:
    """Analyze project for migration opportunities"""
    print("🔍 Analyzing project...")
    migration_state.current_step = "analysis"
    
    try:
        project_path = Path(migration_state.project_path)
        
        # Check if it's a Maven project
        pom_path = project_path / "pom.xml"
        if not pom_path.exists():
            error = "No pom.xml found - not a Maven project"
            migration_state.errors.append(error)
            return {"success": False, "error": error}
        
        # Read and analyze pom.xml
        with open(pom_path, 'r') as f:
            pom_content = f.read()
        
        # Parse pom.xml
        try:
            root = ET.fromstring(pom_content)
            ns = {'maven': 'http://maven.apache.org/POM/4.0.0'}
            
            # Extract properties
            properties = {}
            props_elem = root.find('.//maven:properties', ns)
            if props_elem is not None:
                for prop in props_elem:
                    tag_name = prop.tag.split('}')[-1] if '}' in prop.tag else prop.tag
                    properties[tag_name] = prop.text or ''
            
        except ET.ParseError:
            properties = {}
        
        # Identify Java version
        java_version = "8"  # Default assumption
        if "java.version" in properties:
            version_str = properties["java.version"]
            if "11" in version_str:
                java_version = "11"
            elif "17" in version_str:
                java_version = "17"
            elif "21" in version_str:
                java_version = "21"
        elif "java.version>11" in pom_content:
            java_version = "11"
        elif "java.version>17" in pom_content:
            java_version = "17"
        elif "java.version>21" in pom_content:
            java_version = "21"
        
        # Count dependencies and plugins
        dependency_count = pom_content.count("<dependency>")
        plugin_count = pom_content.count("<plugin>")
        
        # Count Java files
        java_files = list(project_path.rglob("*.java"))
        java_file_count = len(java_files)
        
        # Check for common migration issues
        issues = []
        if java_version in ["8", "11", "17"]:
            issues.append(f"Java {java_version} needs migration to Java 21")
        if "javax." in pom_content:
            issues.append("javax packages need migration to jakarta")
        if "junit" in pom_content.lower() and "jupiter" not in pom_content.lower():
            issues.append("JUnit 4 needs migration to JUnit 5")
        if "spring-boot" in pom_content and "3." not in pom_content:
            issues.append("Spring Boot needs upgrade to 3.x")
        
        # Check Java source files for additional issues
        javax_usage = 0
        junit4_usage = 0
        
        for java_file in java_files[:10]:  # Sample first 10 files
            try:
                with open(java_file, 'r') as f:
                    content = f.read()
                if "import javax." in content:
                    javax_usage += 1
                if "import org.junit.Test" in content or "@Test" in content:
                    junit4_usage += 1
            except:
                continue
        
        if javax_usage > 0:
            issues.append(f"Found javax imports in {javax_usage} Java files")
        if junit4_usage > 0:
            issues.append(f"Found JUnit 4 usage in {junit4_usage} Java files")
        
        migration_state.issues_found = issues
        
        analysis_results = {
            "java_version": java_version,
            "dependency_count": dependency_count,
            "plugin_count": plugin_count,
            "java_file_count": java_file_count,
            "migration_complexity": "simple" if dependency_count < 10 else "moderate" if dependency_count < 50 else "complex",
            "issues_found": issues,
            "properties": properties
        }
        
        print(f"📊 Analysis Results:")
        print(f"   • Java Version: {java_version}")
        print(f"   • Dependencies: {dependency_count}")
        print(f"   • Plugins: {plugin_count}")
        print(f"   • Java Files: {java_file_count}")
        print(f"   • Complexity: {analysis_results['migration_complexity']}")
        print(f"   • Issues Found: {len(issues)}")
        for issue in issues:
            print(f"     - {issue}")
        
        return {"success": True, "analysis": analysis_results}
        
    except Exception as e:
        error = f"Analysis failed: {str(e)}"
        migration_state.errors.append(error)
        return {"success": False, "error": error}

# Run the analysis
analysis_result = analyze_project()
analysis_result

## Step 2: Generate Analysis Prompt

Use the analysis prompt template to create a comprehensive analysis prompt.

In [ ]:
# Generate analysis prompt using our template
if analysis_result["success"]:
    analysis_data = analysis_result["analysis"]
    
    # Read a sample Java file for analysis
    project_path = Path(migration_state.project_path)
    java_files = list(project_path.rglob("*.java"))
    sample_code = "No Java files found"
    
    if java_files:
        try:
            with open(java_files[0], 'r') as f:
                sample_code = f.read()[:2000]  # First 2000 chars
        except:
            sample_code = "Could not read Java file"
    
    analysis_vars = {
        'code': sample_code,
        'java_version': analysis_data['java_version'],
        'dependencies': f"{analysis_data['dependency_count']} dependencies, {analysis_data['plugin_count']} plugins"
    }
    
    # Generate the prompt
    analysis_prompt = prompt_manager.get_prompt('analysis', analysis_vars)
    
    print("🎯 Generated Analysis Prompt:")
    print("="*60)
    print(analysis_prompt[:1000] + "..." if len(analysis_prompt) > 1000 else analysis_prompt)
    print("="*60)
    print(f"📏 Full prompt length: {len(analysis_prompt)} characters")
    
    # Store for potential LLM use
    migration_state.analysis_prompt = analysis_prompt
else:
    print("❌ Cannot generate analysis prompt - analysis failed")

## Step 3: Recipe Identification

Identify appropriate migration recipes based on the analysis.

In [ ]:
def generate_migration_report() -> str:
    """Generate comprehensive migration report"""
    print("📋 Generating migration report...")
    
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    
    report = f"""# Java Migration Report

**Project:** {migration_state.project_path}
**Migration Date:** {datetime.now().isoformat()}
**Final Status:** {'✅ SUCCESS' if migration_state.success else '❌ PARTIAL/FAILED'}
**Current Step:** {migration_state.current_step}

## Executive Summary

This report documents the automated migration of a Java project targeting Java 21 compatibility.
The migration system used YAML-based prompt templates and a structured workflow approach.

## Migration Results

### Issues Identified ({len(migration_state.issues_found)})
{chr(10).join(f"- {issue}" for issue in migration_state.issues_found) if migration_state.issues_found else "- No issues identified"}

### Fixes Applied ({len(migration_state.fixes_applied)})
{chr(10).join(f"- {fix}" for fix in migration_state.fixes_applied) if migration_state.fixes_applied else "- No fixes applied"}

### Errors Encountered ({len(migration_state.errors)})
{chr(10).join(f"- {error}" for error in migration_state.errors) if migration_state.errors else "- No errors encountered"}

## Migration Workflow

The following steps were executed:

1. ✅ **Project Analysis** - Analyzed project structure and identified migration opportunities
2. ✅ **Recipe Identification** - Identified appropriate migration recipes based on analysis
3. ✅ **Recipe Application** - Applied selected migration recipes to the codebase
4. ✅ **Error Detection** - Checked for compilation errors and applied simple fixes
5. ✅ **Testing** - Ran project tests to validate migration success
6. ✅ **Build Validation** - Performed full build validation
7. ✅ **Report Generation** - Generated this comprehensive report

## Prompt Templates Generated

The following prompts were generated and can be used with LLM systems:

### Analysis Prompt
- **Purpose:** Analyze Java code for deprecated APIs and migration issues
- **Variables:** code, java_version, dependencies
- **Generated:** {'✅ Yes' if hasattr(migration_state, 'analysis_prompt') else '❌ No'}

### Recipe Identification Prompt  
- **Purpose:** Identify appropriate migration recipes based on analysis
- **Variables:** analysis_results, available_recipes
- **Generated:** {'✅ Yes' if hasattr(migration_state, 'recipe_prompt') else '❌ No'}

### Code Migration Prompt
- **Purpose:** Apply specific migration recipes to source code
- **Variables:** recipe, source_code, file_path
- **Generated:** {'✅ Yes' if 'migration_prompt' in locals() else '❌ No'}

### Error Fixing Prompt
- **Purpose:** Fix compilation and runtime errors after migration
- **Variables:** error_message, source_code, file_path, applied_recipe, previous_changes
- **Generated:** {'✅ Yes' if 'error_prompt' in locals() else '❌ No'}

### Testing Prompt
- **Purpose:** Create and validate tests for migrated code
- **Variables:** migrated_code, original_code, migration_summary
- **Generated:** {'✅ Yes' if 'test_prompt' in locals() else '❌ No'}

### Build Validation Prompt
- **Purpose:** Validate that migrated project builds and runs correctly
- **Variables:** project_structure, build_config, migration_summary, build_output, test_results
- **Generated:** {'✅ Yes' if 'build_prompt' in locals() else '❌ No'}

## Technical Details

### Project Configuration
- **Maven Project:** {'✅ Yes' if (Path(migration_state.project_path) / "pom.xml").exists() else '❌ No'}
- **Java Files:** {len(list(Path(migration_state.project_path).rglob("*.java")))} found
- **Prompt Manager:** ✅ Integrated with YAML templates
- **Template Count:** {len(prompt_manager.list_templates())} available

### Recommendations

1. **Review Changes:** All automated changes should be reviewed before production deployment
2. **Test Coverage:** Ensure comprehensive test coverage for migrated functionality  
3. **Performance Testing:** Validate performance with Java 21 runtime
4. **Dependency Updates:** Consider updating to latest compatible dependency versions
5. **Documentation:** Update project documentation to reflect Java 21 requirements

### Next Steps

1. Manual review of all applied changes
2. Extended integration testing
3. Performance benchmarking with Java 21
4. Update CI/CD pipeline for Java 21 
5. Deploy to staging environment for validation

## Prompt Template Usage

All generated prompts can be used with LLM systems like Claude, GPT-4, or other language models to:

- Get detailed analysis of migration issues
- Receive specific code fixes and recommendations  
- Generate test cases for validation
- Troubleshoot build and runtime issues
- Create comprehensive migration documentation

## System Information

- **Migration System:** MVP Java Migration System v1.0
- **Prompt Templates:** YAML-based template system
- **Workflow Engine:** Jupyter Notebook with granular control
- **Target Java Version:** 21 LTS
- **Build System:** Apache Maven

---
*Report generated by MVP Java Migration System*
*Timestamp: {timestamp}*
"""
    
    # Save report
    report_path = Path(migration_state.project_path) / f"migration-report-{timestamp}.md"
    with open(report_path, 'w') as f:
        f.write(report)
    
    print(f"📄 Migration report saved: {report_path}")
    return str(report_path)

# Generate the report
report_path = generate_migration_report()

# Display summary
print(f"\n🎯 Migration Summary:")
print(f"   • Project: {migration_state.project_path}")
print(f"   • Success: {'✅ Yes' if migration_state.success else '❌ No'}")
print(f"   • Issues Found: {len(migration_state.issues_found)}")
print(f"   • Fixes Applied: {len(migration_state.fixes_applied)}")
print(f"   • Errors: {len(migration_state.errors)}")
print(f"   • Current Step: {migration_state.current_step}")
print(f"   • Report: {report_path}")

migration_state

## Step 4: Apply Migration Recipes

Apply the identified recipes to the project. You can choose which recipes to apply.

In [ ]:

def identify_migration_recipes(analysis_data: Dict[str, Any]) -> List[Dict[str, Any]]:
    """Identify migration recipes based on analysis"""
    print("🎯 Identifying migration recipes...")
    migration_state.current_step = "recipe_identification"
    
    recipes = []
    
    # Java version migration
    java_version = analysis_data["java_version"]
    if java_version == "8":
        recipes.append({
            "name": "Java8to21Migration",
            "priority": "high",
            "description": "Migrate Java 8 directly to Java 21",
            "type": "java_version",
            "estimated_effort": "high"
        })
    elif java_version == "11":
        recipes.append({
            "name": "Java11to21Migration", 
            "priority": "high",
            "description": "Migrate Java 11 to Java 21",
            "type": "java_version",
            "estimated_effort": "medium"
        })
    elif java_version == "17":
        recipes.append({
            "name": "Java17to21Migration",
            "priority": "high", 
            "description": "Migrate Java 17 to Java 21",
            "type": "java_version",
            "estimated_effort": "low"
        })
    
    # Framework-specific recipes
    for issue in analysis_data["issues_found"]:
        if "javax" in issue.lower():
            recipes.append({
                "name": "JavaxToJakartaMigration",
                "priority": "high",
                "description": "Migrate javax packages to jakarta",
                "type": "framework",
                "estimated_effort": "medium"
            })
        elif "junit 4" in issue.lower():
            recipes.append({
                "name": "JUnit4to5Migration",
                "priority": "medium",
                "description": "Migrate JUnit 4 to JUnit 5",
                "type": "testing",
                "estimated_effort": "medium"
            })
        elif "spring boot" in issue.lower():
            recipes.append({
                "name": "SpringBoot3Migration",
                "priority": "high",
                "description": "Upgrade Spring Boot to 3.x",
                "type": "framework",
                "estimated_effort": "high"
            })
    
    # Always include dependency updates
    recipes.append({
        "name": "DependencyUpdate",
        "priority": "medium",
        "description": "Update dependencies to latest compatible versions",
        "type": "dependencies",
        "estimated_effort": "low"
    })
    
    # Maven plugin updates
    recipes.append({
        "name": "MavenPluginUpdate",
        "priority": "medium",
        "description": "Update Maven plugins for Java 21 compatibility",
        "type": "build",
        "estimated_effort": "low"
    })
    
    # Remove duplicates and sort by priority
    unique_recipes = []
    seen_names = set()
    for recipe in recipes:
        if recipe["name"] not in seen_names:
            unique_recipes.append(recipe)
            seen_names.add(recipe["name"])
    
    # Sort by priority (high first)
    priority_order = {"high": 0, "medium": 1, "low": 2}
    unique_recipes.sort(key=lambda x: priority_order.get(x["priority"], 3))
    
    print(f"🎯 Identified {len(unique_recipes)} migration recipes:")
    for i, recipe in enumerate(unique_recipes, 1):
        print(f"   {i}. {recipe['name']} ({recipe['priority']} priority)")
        print(f"      └─ {recipe['description']}")
    
    return unique_recipes

# Identify recipes
if analysis_result["success"]:
    identified_recipes = identify_migration_recipes(analysis_result["analysis"])
    
    # Generate recipe identification prompt
    recipe_vars = {
        'analysis_results': json.dumps(analysis_result["analysis"], indent=2),
        'available_recipes': json.dumps([r["name"] for r in identified_recipes], indent=2)
    }
    
    recipe_prompt = prompt_manager.get_prompt('recipe_identification', recipe_vars)
    migration_state.recipe_prompt = recipe_prompt
    
    print(f"\n🎯 Recipe identification prompt generated ({len(recipe_prompt)} chars)")
    
    identified_recipes
else:
    print("❌ Cannot identify recipes - analysis failed")
    identified_recipes = []

## Step 6: Testing and Validation

Run tests to ensure the migration works correctly.

In [ ]:
def check_compilation() -> Dict[str, Any]:
    """Check if project compiles successfully"""
    print("🔨 Checking compilation...")
    migration_state.current_step = "error_detection"
    
    try:
        project_path = Path(migration_state.project_path)
        
        # Run Maven compile
        result = subprocess.run(
            ["mvn", "compile", "-f", str(project_path)],
            capture_output=True, text=True, timeout=120,
            cwd=str(project_path)
        )
        
        if result.returncode == 0:
            print("✅ Compilation successful!")
            return {
                "success": True,
                "errors": [],
                "warnings": [],
                "output": result.stdout
            }
        else:
            print("⚠️ Compilation errors found")
            
            # Parse errors from output
            error_lines = []
            warning_lines = []
            
            for line in result.stderr.split('\n'):
                if '[ERROR]' in line:
                    error_lines.append(line.strip())
                elif '[WARNING]' in line:
                    warning_lines.append(line.strip())
            
            print(f"   Found {len(error_lines)} errors, {len(warning_lines)} warnings")
            
            return {
                "success": False,
                "errors": error_lines,
                "warnings": warning_lines,
                "output": result.stderr,
                "full_output": result.stdout + result.stderr
            }
    
    except subprocess.TimeoutExpired:
        error_msg = "Maven compile timed out after 120 seconds"
        migration_state.errors.append(error_msg)
        return {"success": False, "error": error_msg}
    except Exception as e:
        error_msg = f"Compilation check failed: {str(e)}"
        migration_state.errors.append(error_msg)
        return {"success": False, "error": error_msg}

def apply_simple_fixes(compilation_result: Dict[str, Any]) -> Dict[str, Any]:
    """Apply simple automated fixes for common issues"""
    print("🔧 Applying simple fixes...")
    
    if compilation_result.get("success", False):
        return {"success": True, "fixes_applied": 0, "message": "No fixes needed - compilation successful"}
    
    try:
        project_path = Path(migration_state.project_path)
        java_files = list(project_path.rglob("*.java"))
        
        fixes_applied = 0
        
        # Common fixes for Java 21 migration
        common_fixes = {
            "new Integer(": "Integer.valueOf(",
            "new Long(": "Long.valueOf(",
            "new Double(": "Double.valueOf(",
            "new Boolean(": "Boolean.valueOf(",
            "new Float(": "Float.valueOf(",
            "new Short(": "Short.valueOf(",
            "new Byte(": "Byte.valueOf(",
        }
        
        for java_file in java_files:
            try:
                with open(java_file, 'r') as f:
                    content = f.read()
                
                original_content = content
                
                # Apply common fixes
                for old_pattern, new_pattern in common_fixes.items():
                    if old_pattern in content:
                        content = content.replace(old_pattern, new_pattern)
                
                if content != original_content:
                    with open(java_file, 'w') as f:
                        f.write(content)
                    fixes_applied += 1
                    print(f"      🔧 Applied fixes to {java_file.name}")
                    
            except Exception as e:
                print(f"      ⚠️ Error fixing {java_file}: {e}")
        
        if fixes_applied > 0:
            migration_state.fixes_applied.append(f"Applied simple fixes to {fixes_applied} Java files")
        
        return {
            "success": True,
            "fixes_applied": fixes_applied,
            "message": f"Applied simple fixes to {fixes_applied} files"
        }
        
    except Exception as e:
        return {"success": False, "error": str(e)}

# Check compilation
compilation_result = check_compilation()

if not compilation_result["success"] and "errors" in compilation_result:
    print(f"\n📋 Found {len(compilation_result['errors'])} compilation errors:")
    for i, error in enumerate(compilation_result["errors"][:5], 1):  # Show first 5
        print(f"   {i}. {error}")
    
    if len(compilation_result["errors"]) > 5:
        print(f"   ... and {len(compilation_result['errors']) - 5} more errors")
    
    # Apply simple fixes
    fix_result = apply_simple_fixes(compilation_result)
    print(f"\n🔧 Fix result: {fix_result['message']}")
    
    # Generate error fixing prompt
    if compilation_result.get("full_output"):
        error_vars = {
            'error_message': compilation_result["full_output"][:2000],  # Limit size
            'source_code': "Post-migration Java code",
            'file_path': str(project_path),
            'applied_recipe': "Java version migration",
            'previous_changes': json.dumps(migration_state.fixes_applied)
        }
        
        error_prompt = prompt_manager.get_prompt('error_fixing', error_vars)
        print(f"🎯 Error fixing prompt generated ({len(error_prompt)} chars)")

compilation_result

## Step 5: Error Detection and Fixing

Check for compilation errors and apply fixes.

In [ ]:
# Apply selected recipes (you can choose which ones to run)
# Example: Apply Java version migration

if 'identified_recipes' in locals() and identified_recipes:
    # Apply the Java version migration recipe
    java_recipe = next((r for r in identified_recipes if "Java" in r['name'] and "Migration" in r['name']), None)
    
    if java_recipe:
        print(f"🔧 Applying: {java_recipe['name']}")
        result = apply_recipe(java_recipe)
        
        if result["success"]:
            migration_state.fixes_applied.append(f"Applied {java_recipe['name']}: {result['changes']}")
            print(f"✅ Success: {result['changes']}")
        else:
            migration_state.errors.append(f"Failed {java_recipe['name']}: {result['error']}")
            print(f"❌ Failed: {result['error']}")
            
        # Generate code migration prompt for this recipe
        migration_vars = {
            'recipe': json.dumps(java_recipe),
            'source_code': "Java version migration applied",
            'file_path': str(Path(migration_state.project_path) / "pom.xml")
        }
        
        migration_prompt = prompt_manager.get_prompt('code_migration', migration_vars)
        print(f"\n🎯 Code migration prompt generated ({len(migration_prompt)} chars)")
        
    else:
        print("❌ No Java migration recipe found")
else:
    print("❌ No recipes identified. Run the recipe identification step first.")

In [ ]:
def apply_recipe(recipe: Dict[str, Any]) -> Dict[str, Any]:
    """Apply a single migration recipe"""
    print(f"🔧 Applying recipe: {recipe['name']}")
    
    try:
        project_path = Path(migration_state.project_path)
        
        if recipe['name'] == "Java8to21Migration" or recipe['name'] == "Java11to21Migration" or recipe['name'] == "Java17to21Migration":
            return apply_java_version_migration(project_path)
        elif recipe['name'] == "JavaxToJakartaMigration":
            return apply_javax_to_jakarta_migration(project_path)
        elif recipe['name'] == "JUnit4to5Migration":
            return apply_junit_migration(project_path)
        elif recipe['name'] == "DependencyUpdate":
            return apply_dependency_updates(project_path)
        elif recipe['name'] == "MavenPluginUpdate":
            return apply_maven_plugin_updates(project_path)
        else:
            return {"success": False, "error": f"Unknown recipe: {recipe['name']}"}
            
    except Exception as e:
        return {"success": False, "error": str(e)}

def apply_java_version_migration(project_path: Path) -> Dict[str, Any]:
    """Migrate Java version to 21"""
    try:
        print("      ☕ Updating Java version to 21...")
        pom_path = project_path / "pom.xml"
        
        with open(pom_path, 'r') as f:
            content = f.read()
        
        original_content = content
        
        # Update Java version to 21
        replacements = [
            ("<java.version>1.8</java.version>", "<java.version>21</java.version>"),
            ("<java.version>8</java.version>", "<java.version>21</java.version>"),
            ("<java.version>11</java.version>", "<java.version>21</java.version>"),
            ("<java.version>17</java.version>", "<java.version>21</java.version>"),
            ("java.version>1.8", "java.version>21"),
            ("java.version>8", "java.version>21"),
            ("java.version>11", "java.version>21"),
            ("java.version>17", "java.version>21"),
        ]
        
        for old, new in replacements:
            if old in content:
                content = content.replace(old, new)
                break
        
        if content != original_content:
            with open(pom_path, 'w') as f:
                f.write(content)
            return {"success": True, "changes": "Java version updated to 21 in pom.xml"}
        else:
            return {"success": True, "changes": "Java version already at 21"}
        
    except Exception as e:
        return {"success": False, "error": str(e)}

def apply_javax_to_jakarta_migration(project_path: Path) -> Dict[str, Any]:
    """Migrate javax packages to jakarta"""
    try:
        print("      📦 Migrating javax to jakarta...")
        
        java_files = list(project_path.rglob("*.java"))
        changes_made = 0
        
        javax_to_jakarta = {
            "import javax.servlet": "import jakarta.servlet",
            "import javax.persistence": "import jakarta.persistence",
            "import javax.validation": "import jakarta.validation",
            "import javax.annotation": "import jakarta.annotation",
            "import javax.inject": "import jakarta.inject",
            "import javax.transaction": "import jakarta.transaction",
        }
        
        for java_file in java_files:
            try:
                with open(java_file, 'r') as f:
                    content = f.read()
                
                original_content = content
                
                for old_import, new_import in javax_to_jakarta.items():
                    if old_import in content:
                        content = content.replace(old_import, new_import)
                
                if content != original_content:
                    with open(java_file, 'w') as f:
                        f.write(content)
                    changes_made += 1
                    
            except Exception as e:
                print(f"        ⚠️ Error updating {java_file}: {e}")
        
        return {"success": True, "changes": f"Updated {changes_made} Java files for javax→jakarta migration"}
        
    except Exception as e:
        return {"success": False, "error": str(e)}

def apply_junit_migration(project_path: Path) -> Dict[str, Any]:
    """Migrate JUnit 4 to JUnit 5"""
    try:
        print("      🧪 Migrating JUnit 4 to JUnit 5...")
        
        # Find test files
        test_patterns = ["**/src/test/**/*.java", "**/test/**/*.java"]
        test_files = []
        for pattern in test_patterns:
            test_files.extend(project_path.glob(pattern))
        
        changes_made = 0
        
        junit_migrations = {
            "import org.junit.Test;": "import org.junit.jupiter.api.Test;",
            "import org.junit.Before;": "import org.junit.jupiter.api.BeforeEach;",
            "import org.junit.After;": "import org.junit.jupiter.api.AfterEach;",
            "import org.junit.BeforeClass;": "import org.junit.jupiter.api.BeforeAll;",
            "import org.junit.AfterClass;": "import org.junit.jupiter.api.AfterAll;",
            "import org.junit.Assert;": "import org.junit.jupiter.api.Assertions;",
            "@Before\\n": "@BeforeEach\\n",
            "@After\\n": "@AfterEach\\n",
            "@BeforeClass": "@BeforeAll",
            "@AfterClass": "@AfterAll",
            "Assert.assertEquals": "Assertions.assertEquals",
            "Assert.assertTrue": "Assertions.assertTrue",
            "Assert.assertFalse": "Assertions.assertFalse",
            "Assert.assertNull": "Assertions.assertNull",
            "Assert.assertNotNull": "Assertions.assertNotNull",
        }
        
        for test_file in test_files:
            try:
                with open(test_file, 'r') as f:
                    content = f.read()
                
                original_content = content
                
                for old_pattern, new_pattern in junit_migrations.items():
                    if old_pattern in content:
                        content = content.replace(old_pattern, new_pattern)
                
                if content != original_content:
                    with open(test_file, 'w') as f:
                        f.write(content)
                    changes_made += 1
                    
            except Exception as e:
                print(f"        ⚠️ Error updating {test_file}: {e}")
        
        return {"success": True, "changes": f"Updated {changes_made} test files for JUnit 4→5 migration"}
        
    except Exception as e:
        return {"success": False, "error": str(e)}

def apply_dependency_updates(project_path: Path) -> Dict[str, Any]:
    """Update dependencies (simplified version)"""
    try:
        print("      📦 Updating dependencies...")
        # This is a placeholder - in a real system you'd use Maven Central API
        return {"success": True, "changes": "Dependencies analyzed for updates (placeholder)"}
    except Exception as e:
        return {"success": False, "error": str(e)}

def apply_maven_plugin_updates(project_path: Path) -> Dict[str, Any]:
    """Update Maven plugins for Java 21 compatibility"""
    try:
        print("      🔧 Updating Maven plugins...")
        pom_path = project_path / "pom.xml"
        
        with open(pom_path, 'r') as f:
            content = f.read()
        
        original_content = content
        
        # Update common Maven plugins for Java 21
        plugin_updates = {
            "<version>3.8.1</version>": "<version>3.11.0</version>",  # compiler plugin
            "<version>3.0.0-M5</version>": "<version>3.0.0</version>",  # surefire plugin
            "<maven.compiler.source>8</maven.compiler.source>": "<maven.compiler.source>21</maven.compiler.source>",
            "<maven.compiler.target>8</maven.compiler.target>": "<maven.compiler.target>21</maven.compiler.target>",
            "<maven.compiler.source>11</maven.compiler.source>": "<maven.compiler.source>21</maven.compiler.source>",
            "<maven.compiler.target>11</maven.compiler.target>": "<maven.compiler.target>21</maven.compiler.target>",
        }
        
        changes_made = 0
        for old_version, new_version in plugin_updates.items():
            if old_version in content:
                content = content.replace(old_version, new_version)
                changes_made += 1
        
        if content != original_content:
            with open(pom_path, 'w') as f:
                f.write(content)
            return {"success": True, "changes": f"Updated {changes_made} Maven plugin configurations"}
        else:
            return {"success": True, "changes": "Maven plugins already up to date"}
        
    except Exception as e:
        return {"success": False, "error": str(e)}

print("🔧 Recipe application functions defined. Ready to apply recipes!")
print("📋 Available recipes to apply:")
if 'identified_recipes' in locals():
    for i, recipe in enumerate(identified_recipes):
        print(f"   {i+1}. {recipe['name']} - {recipe['description']}")
else:
    print("   Run the recipe identification step first!")

## Step 7: Migration Report

Generate a comprehensive migration report with all results and prompts.


In [ ]:
def run_tests() -> Dict[str, Any]:
    """Run project tests"""
    print("🧪 Running tests...")
    migration_state.current_step = "testing"
    
    try:
        project_path = Path(migration_state.project_path)
        
        # Run Maven test
        result = subprocess.run(
            ["mvn", "test", "-f", str(project_path)],
            capture_output=True, text=True, timeout=300,
            cwd=str(project_path)
        )
        
        if result.returncode == 0:
            print("✅ All tests passed!")
            
            # Parse test results
            test_summary = "Tests completed successfully"
            for line in result.stdout.split('\n'):
                if 'Tests run:' in line:
                    test_summary = line.strip()
                    break
            
            return {
                "success": True,
                "test_summary": test_summary,
                "output": result.stdout
            }
        else:
            print("⚠️ Some tests failed")
            
            # Parse test failures
            failure_lines = []
            for line in result.stdout.split('\n'):
                if 'FAILURE' in line or 'ERROR' in line or 'Failed' in line:
                    failure_lines.append(line.strip())
            
            print(f"   Found {len(failure_lines)} test failures")
            
            return {
                "success": False,
                "failures": failure_lines,
                "output": result.stdout,
                "full_output": result.stdout + result.stderr
            }
    
    except subprocess.TimeoutExpired:
        error_msg = "Maven tests timed out after 300 seconds"
        migration_state.errors.append(error_msg)
        return {"success": False, "error": error_msg}
    except Exception as e:
        error_msg = f"Test execution failed: {str(e)}"
        migration_state.errors.append(error_msg)
        return {"success": False, "error": error_msg}

def run_full_build() -> Dict[str, Any]:
    """Run full Maven build including tests"""
    print("🏗️ Running full build...")
    migration_state.current_step = "build_validation"
    
    try:
        project_path = Path(migration_state.project_path)
        
        # Run full Maven build
        result = subprocess.run(
            ["mvn", "clean", "compile", "test", "-f", str(project_path)],
            capture_output=True, text=True, timeout=600,
            cwd=str(project_path)
        )
        
        if result.returncode == 0:
            print("✅ Full build successful!")
            migration_state.success = True
            
            return {
                "success": True,
                "output": result.stdout
            }
        else:
            print("❌ Build failed")
            
            return {
                "success": False,
                "output": result.stderr,
                "full_output": result.stdout + result.stderr
            }
    
    except subprocess.TimeoutExpired:
        error_msg = "Maven build timed out after 600 seconds"
        migration_state.errors.append(error_msg)
        return {"success": False, "error": error_msg}
    except Exception as e:
        error_msg = f"Build execution failed: {str(e)}"
        migration_state.errors.append(error_msg)
        return {"success": False, "error": error_msg}

# Run tests
test_result = run_tests()

if test_result["success"]:
    print(f"✅ Test Summary: {test_result.get('test_summary', 'Tests passed')}")
else:
    if "failures" in test_result:
        print(f"\n📋 Found {len(test_result['failures'])} test failures:")
        for i, failure in enumerate(test_result["failures"][:3], 1):  # Show first 3
            print(f"   {i}. {failure}")
    
    # Generate testing prompt for failures
    if test_result.get("full_output"):
        test_vars = {
            'migrated_code': "Post-migration code",
            'original_code': "Pre-migration code",
            'migration_summary': json.dumps({
                "fixes_applied": migration_state.fixes_applied,
                "issues_found": migration_state.issues_found
            })
        }
        
        test_prompt = prompt_manager.get_prompt('testing', test_vars)
        print(f"🎯 Testing prompt generated ({len(test_prompt)} chars)")

print(f"\n🏗️ Running full build validation...")
build_result = run_full_build()

if build_result["success"]:
    print("🎉 Migration completed successfully!")
else:
    print("⚠️ Build validation failed")
    
    # Generate build validation prompt
    build_vars = {
        'project_structure': str(Path(migration_state.project_path)),
        'build_config': "Maven pom.xml configuration",
        'migration_summary': json.dumps({
            "fixes_applied": migration_state.fixes_applied,
            "issues_found": migration_state.issues_found,
            "errors": migration_state.errors
        }),
        'build_output': build_result.get("full_output", "No build output"),
        'test_results': test_result.get("output", "No test results")
    }
    
    build_prompt = prompt_manager.get_prompt('build_validation', build_vars)
    print(f"🎯 Build validation prompt generated ({len(build_prompt)} chars)")

{"test_result": test_result, "build_result": build_result}

## Step 8: Human Escalation and Manual Review

If you need human input for complex issues, use this section.

In [ ]:
def request_human_escalation(
    escalation_reason: str,
    problem_description: str,
    current_code: str = "",
    error_details: str = "",
    specific_questions: str = "",
    suggested_actions: str = ""
) -> str:
    """Generate human escalation request with full context"""
    
    escalation_vars = {
        'project_name': Path(migration_state.project_path).name,
        'file_path': migration_state.project_path,
        'current_step': migration_state.current_step,
        'escalation_reason': escalation_reason,
        'problem_description': problem_description,
        'attempted_solutions': json.dumps(migration_state.fixes_applied),
        'current_code': current_code,
        'error_details': error_details,
        'target_version': '21',
        'applied_recipes': json.dumps(migration_state.fixes_applied),
        'previous_changes': json.dumps(migration_state.fixes_applied),
        'specific_questions': specific_questions,
        'suggested_actions': suggested_actions
    }
    
    # Generate human escalation prompt
    escalation_prompt = prompt_manager.get_prompt('human_escalation', escalation_vars)
    
    print("🚨 HUMAN ESCALATION REQUEST")
    print("="*60)
    print(escalation_prompt)
    print("="*60)
    
    return escalation_prompt

# Example escalation usage (uncomment and modify as needed)
"""
escalation_prompt = request_human_escalation(
    escalation_reason="Complex compilation errors after migration",
    problem_description="Multiple compilation errors that automated fixes couldn't resolve",
    error_details="Compilation errors from Maven output",
    specific_questions="Should we rollback changes or proceed with manual fixes?",
    suggested_actions="1. Manual code review 2. Selective rollback 3. Custom migration recipe"
)
"""

print("🔧 Human escalation system ready.")
print("📋 Use request_human_escalation() function when you need human input.")
print("\n💡 Typical escalation scenarios:")
print("   • Complex compilation errors")
print("   • Test failures requiring domain knowledge") 
print("   • Framework-specific migration issues")
print("   • Performance or compatibility concerns")
print("   • Custom business logic migration needs")
